# 🤖 Jarvis — Full AI Assistant with 3D Vision

The complete Jarvis experience:
- 🎙️ Talk to Jarvis by voice
- 🧠 Jarvis thinks with GPT-4o
- 🔊 Jarvis speaks back to you
- 📸 Say **"make a 3D model"** → Upload image → Download `.obj` file

### Setup checklist:
- ✅ Runtime set to **T4 GPU** (Runtime → Change runtime type)
- ✅ `OPENAI_API_KEY` added to **Colab Secrets** (🔑 key icon on left sidebar)
- ✅ Ran `01_setup.ipynb` at least once

---

In [ ]:
# ── CELL 1: Install & Setup ──────────────────────────────────────────────────
import os, sys, base64, glob, shutil
from google.colab import userdata, files, output
from IPython.display import Javascript, Audio, display as ipy_display
from PIL import Image
import matplotlib.pyplot as plt

# Load API key
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Ensure TripoSR is present
if not os.path.exists('/content/TripoSR'):
    print('Installing TripoSR (first time only)...')
    os.system('git clone https://github.com/VAST-AI-Research/TripoSR /content/TripoSR')
    os.system('pip install -q -r /content/TripoSR/requirements.txt')

# Install remaining packages
os.system('pip install -q openai openai-whisper gTTS rembg')

import openai
import whisper
import torch
from gtts import gTTS
from rembg import remove

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('Loading Whisper model...')
whisper_model = whisper.load_model('base')

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])

print('Jarvis is online and ready.')

In [ ]:
# ── CELL 2: Core Functions ───────────────────────────────────────────────────

# --- Conversation memory ---
conversation_history = [
    {
        'role': 'system',
        'content': (
            'You are Jarvis, a highly intelligent personal AI assistant inspired by Iron Man. '
            'You are calm, professional, and concise. '
            'When the user asks you to create or generate a 3D model of something, '
            'respond with exactly: TRIGGER_3D_MODEL followed by your verbal response. '
            'Example: TRIGGER_3D_MODEL Understood. Please upload an image of the object.'
            'For all other queries respond normally in under 3 sentences.'
        )
    }
]

# --- Browser mic recorder ---
RECORD_JS = """
async function recordAudio(durationMs) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await new Promise(r => setTimeout(r, durationMs));
  recorder.stop();
  await new Promise(r => recorder.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks, { type: 'audio/webm' });
  const reader = new FileReader();
  return new Promise(resolve => {
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
}
"""

def record_voice(seconds=6):
    ipy_display(Javascript(RECORD_JS))
    print(f'Listening for {seconds} seconds... Speak now!')
    data_url = output.eval_js(f'recordAudio({seconds * 1000})')
    audio_bytes = base64.b64decode(data_url.split(',')[1])
    path = '/tmp/jarvis_in.webm'
    with open(path, 'wb') as f:
        f.write(audio_bytes)
    return path

def transcribe(audio_path):
    result = whisper_model.transcribe(audio_path)
    return result['text'].strip()

def ask_jarvis(user_text):
    conversation_history.append({'role': 'user', 'content': user_text})
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=conversation_history,
        max_tokens=300
    )
    reply = response.choices[0].message.content.strip()
    conversation_history.append({'role': 'assistant', 'content': reply})
    return reply

def speak(text):
    clean_text = text.replace('TRIGGER_3D_MODEL', '').strip()
    tts = gTTS(text=clean_text, lang='en', slow=False)
    path = '/tmp/jarvis_out.mp3'
    tts.save(path)
    ipy_display(Audio(path, autoplay=True))
    return clean_text

def generate_3d(image_path):
    """Remove background then run TripoSR on an image."""
    out_dir = '/content/3d_output'
    os.makedirs(out_dir, exist_ok=True)
    # Remove background
    print('Removing background...')
    clean_path = '/content/model_input.png'
    with open(image_path, 'rb') as i, open(clean_path, 'wb') as o:
        o.write(remove(i.read()))
    # Generate 3D
    print('Generating 3D model (this takes ~2 minutes)...')
    ret = os.system(f'python /content/TripoSR/run.py {clean_path} --output-dir {out_dir} --device cuda --render')
    if ret != 0:
        print('3D generation encountered an error. Trying without --render flag...')
        os.system(f'python /content/TripoSR/run.py {clean_path} --output-dir {out_dir} --device cuda')
    return out_dir

print('All functions loaded. Jarvis is ready.')

In [ ]:
# ── CELL 3: 🎙️ TALK TO JARVIS ───────────────────────────────────────────────
# Run this cell every time you want to speak to Jarvis

print('=' * 50)
print('JARVIS LISTENING...')
print('=' * 50)

audio_path = record_voice(seconds=6)
user_text = transcribe(audio_path)
print(f'\nYou: "{user_text}"')

if not user_text:
    print('Nothing heard. Run the cell again.')
else:
    jarvis_reply = ask_jarvis(user_text)
    print(f'Jarvis (raw): "{jarvis_reply}"')

    spoken = speak(jarvis_reply)
    print(f'Jarvis: "{spoken}"')

    # Check if Jarvis wants to make a 3D model
    if 'TRIGGER_3D_MODEL' in jarvis_reply:
        print('\n3D MODEL MODE ACTIVATED')
        print('Upload an image of the object you want to make a 3D model of:')
        uploaded = files.upload()
        if uploaded:
            img_name = list(uploaded.keys())[0]
            img_path = f'/content/{img_name}'

            # Preview
            plt.figure(figsize=(4, 4))
            plt.imshow(Image.open(img_path))
            plt.axis('off')
            plt.title('Input image')
            plt.show()

            # Generate 3D
            out_dir = generate_3d(img_path)

            # Show renders
            renders = glob.glob(f'{out_dir}/**/*.png', recursive=True)
            if renders:
                cols = min(len(renders), 4)
                fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
                if cols == 1:
                    axes = [axes]
                for i, r in enumerate(renders[:4]):
                    axes[i].imshow(Image.open(r))
                    axes[i].axis('off')
                    axes[i].set_title(f'View {i+1}')
                plt.tight_layout()
                plt.show()

            # Download
            model_files = (
                glob.glob(f'{out_dir}/**/*.obj', recursive=True) +
                glob.glob(f'{out_dir}/**/*.glb', recursive=True)
            )
            if model_files:
                speak('Your 3D model is ready. Downloading now.')
                for mf in model_files:
                    print(f'Downloading: {os.path.basename(mf)}')
                    files.download(mf)
            else:
                speak('I could not generate the model. Please try with a clearer image.')

In [ ]:
# ── CELL 4: ✍️ TYPE to Jarvis (no mic needed) ───────────────────────────────

user_input = input('You: ')
jarvis_reply = ask_jarvis(user_input)
spoken = speak(jarvis_reply)
print(f'Jarvis: "{spoken}"')

if 'TRIGGER_3D_MODEL' in jarvis_reply:
    print('\nUpload an image to generate its 3D model:')
    uploaded = files.upload()
    if uploaded:
        img_name = list(uploaded.keys())[0]
        out_dir = generate_3d(f'/content/{img_name}')
        model_files = (
            glob.glob(f'{out_dir}/**/*.obj', recursive=True) +
            glob.glob(f'{out_dir}/**/*.glb', recursive=True)
        )
        for mf in model_files:
            files.download(mf)
        speak('3D model downloaded.')

In [ ]:
# ── CELL 5: 📸 Directly upload image → 3D (no voice needed) ─────────────────

print('Upload any image to generate a 3D model directly:')
uploaded = files.upload()

if uploaded:
    img_name = list(uploaded.keys())[0]
    img_path = f'/content/{img_name}'

    plt.figure(figsize=(4, 4))
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.title('Input')
    plt.show()

    out_dir = generate_3d(img_path)

    renders = glob.glob(f'{out_dir}/**/*.png', recursive=True)
    if renders:
        cols = min(len(renders), 4)
        fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
        if cols == 1: axes = [axes]
        for i, r in enumerate(renders[:4]):
            axes[i].imshow(Image.open(r))
            axes[i].axis('off')
        plt.tight_layout()
        plt.show()

    model_files = (
        glob.glob(f'{out_dir}/**/*.obj', recursive=True) +
        glob.glob(f'{out_dir}/**/*.glb', recursive=True)
    )
    if model_files:
        speak('Your 3D model is ready.')
        for mf in model_files:
            print(f'Downloading: {os.path.basename(mf)}')
            files.download(mf)
    else:
        print('No model files found. Check errors above.')